In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install dependencies
!pip install transformers datasets torch --quiet

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, get_linear_schedule_with_warmup
from datasets import load_dataset
from tqdm import tqdm
import os

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Configuration
MODEL_NAME = 'distilbert-base-uncased'
NUM_LABELS = 4
MAX_LENGTH = 128
BATCH_SIZE = 32  # Larger batch size with GPU
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
SAMPLE_FRACTION = 1.0  # Use full data (1.0) or reduce for faster training

LABEL_NAMES = {0: 'World', 1: 'Sports', 2: 'Business', 3: 'Sci/Tech'}

In [ ]:
# Dataset class
class AGNewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
# Load data
print("Loading AG News dataset...")
dataset = load_dataset("ag_news")

train_texts = dataset['train']['text']
train_labels = dataset['train']['label']
test_texts = dataset['test']['text']
test_labels = dataset['test']['label']

# Sample if needed
if SAMPLE_FRACTION < 1.0:
    sample_size = int(len(train_texts) * SAMPLE_FRACTION)
    np.random.seed(42)
    indices = np.random.choice(len(train_texts), sample_size, replace=False)
    train_texts = [train_texts[i] for i in indices]
    train_labels = [train_labels[i] for i in indices]
    print(f"Sampled {SAMPLE_FRACTION*100:.0f}% of training data")

# Validation split
split_idx = int(0.9 * len(train_texts))
val_texts = train_texts[split_idx:]
val_labels = train_labels[split_idx:]
train_texts = train_texts[:split_idx]
train_labels = train_labels[:split_idx]

print(f"✓ Training samples: {len(train_texts)}")
print(f"✓ Validation samples: {len(val_texts)}")
print(f"✓ Test samples: {len(test_texts)}")

In [ ]:
# Initialize tokenizer and model
print("Loading model...")
tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)
model = DistilBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NUM_LABELS)
model = model.to(device)
print("✓ Model loaded")

In [ ]:
# Create dataloaders
train_dataset = AGNewsDataset(train_texts, train_labels, tokenizer, MAX_LENGTH)
val_dataset = AGNewsDataset(val_texts, val_labels, tokenizer, MAX_LENGTH)
test_dataset = AGNewsDataset(test_texts, test_labels, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print(f"✓ Created dataloaders")

In [ ]:
# Setup optimizer and scheduler
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
total_steps = len(train_loader) * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)
print(f"✓ Setup optimizer (total steps: {total_steps})")

In [ ]:
# Training loop
print("\n" + "="*60)
print("TRAINING TRANSFORMER MODEL")
print("="*60)

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print("-" * 40)
    
    # Training
    model.train()
    train_loss = 0
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    for batch in progress_bar:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        train_loss += loss.item()
        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_train_loss = train_loss / len(train_loader)
    
    # Validation
    model.eval()
    val_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validating"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            val_loss += outputs.loss.item()
            
            _, predicted = torch.max(outputs.logits, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    avg_val_loss = val_loss / len(val_loader)
    val_accuracy = correct / total
    
    print(f"Train Loss: {avg_train_loss:.4f}")
    print(f"Val Loss: {avg_val_loss:.4f}")
    print(f"Val Accuracy: {val_accuracy:.4f}")

print("\n✓ Training complete!")

In [ ]:
# Evaluate on test set
print("\n" + "="*60)
print("EVALUATING ON TEST SET")
print("="*60)

model.eval()
correct = 0
total = 0
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Testing"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        _, predicted = torch.max(outputs.logits, 1)
        
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

test_accuracy = correct / total
print(f"\n✓ Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

In [ ]:
# Save model
os.makedirs('transformer_model', exist_ok=True)
model.save_pretrained('transformer_model')
tokenizer.save_pretrained('transformer_model')
print("✓ Model saved to 'transformer_model/' folder")

In [ ]:
# Zip the model for download
!zip -r transformer_model.zip transformer_model/
print("\n✓ Created transformer_model.zip")
print("\n📥 Download the zip file and extract to your local 'models/transformer/' folder")

In [ ]:
# Download link (for Colab)
from google.colab import files
files.download('transformer_model.zip')